In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import numpy as np
import pandas as pd

from sklearn.linear_model import Ridge
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_log_error
from statsmodels.tsa.deterministic import CalendarFourier, DeterministicProcess, Fourier

import warnings
warnings.filterwarnings("ignore")

In [ ]:
path = '../input/store-sales-time-series-forecasting/'

In [ ]:
df_train = pd.read_csv(path + 'train.csv',
                       usecols=['store_nbr', 'family', 'date', 'sales'],
                       dtype={'store_nbr': 'category', 'family': 'category', 'sales': 'float32'},
                       parse_dates=['date'], infer_datetime_format=True)

df_train.date = df_train.date.dt.to_period('D')
df_train = df_train.set_index(['store_nbr', 'family', 'date']).sort_index()

In [ ]:
df_test= pd.read_csv(path + 'test.csv',
                       usecols=['store_nbr', 'family', 'date'],
                       dtype={'store_nbr': 'category', 'family': 'category'},
                       parse_dates=['date'], infer_datetime_format=True)

df_test.date = df_test.date.dt.to_period('D')
df_test = df_test.set_index(['store_nbr', 'family', 'date']).sort_index()

In [ ]:
y = df_train.unstack(['store_nbr', 'family']).loc["2017"]


In [ ]:
y

In [ ]:

# Create training data
fourier = CalendarFourier(freq='M', order=4)
dp = DeterministicProcess(
    index=y.index,
    constant=True,
    order=2,
    seasonal=True,
    additional_terms=[fourier],
    drop=True,
)
X = dp.in_sample()
X['NewYear'] = (X.index.dayofyear == 1)


In [ ]:
X

In [ ]:
# Create class for two algorithms
class BoostedHybrid:
    def __init__(self, model_1, model_2):
        self.model_1 = model_1
        self.model_2 = model_2
        self.y_columns = None  # store column names from fit method

In [ ]:
# Add 'predict' method to the BoostedHybrid class 
def predict(self, X):
    y_pred1 = pd.DataFrame(self.model_1.predict(X),index=X.index, columns=self.y_columns)
    y_pred2 = pd.DataFrame(self.model_2.predict(X),index=X.index, columns=self.y_columns)     
    y_pred = y_pred1 + y_pred2 
    
    return y_pred

# Add method to class
BoostedHybrid.predict = predict

In [ ]:
# Add 'fit' method to the BoostedHybrid class 
def fit(self, X, y):

    self.model_1.fit(X,y)
    y_fit = pd.DataFrame(self.model_1.predict(X),index=X.index, columns=y.columns)
    y_resid = y - y_fit
    self.model_2.fit(X, y_resid)

    # Save column names for predict method
    self.y_columns = y.columns

# Add method to class
BoostedHybrid.fit = fit

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


# مدل اول: Ridge با استانداردسازی
ridge_pipeline = make_pipeline(StandardScaler(), Ridge(alpha=0.5))

# مدل ترکیبی
model = BoostedHybrid(
    model_1=ridge_pipeline,
    model_2=RandomForestRegressor(n_estimators=250, random_state=0)
)

model.fit(X, y)
y_pred = model.predict(X)
y_pred = y_pred.clip(0.0)


In [ ]:
from sklearn.metrics import mean_squared_log_error
import numpy as np

# اطمینان از اینکه ایندکس‌ها هم‌راستا هستند
y_pred_stacked = y_pred.stack(['store_nbr', 'family']).reset_index()
y_target_stacked = y.stack(['store_nbr', 'family']).reset_index().copy()

# اتصال پیش‌بینی‌ها به مقدار واقعی
y_target_stacked['sales_pred'] = y_pred_stacked['sales'].values

# محاسبه RMSLE برای هر family
rmsle_per_family = y_target_stacked.groupby('family').apply(
    lambda a: np.sqrt(mean_squared_log_error(a['sales'], a['sales_pred']))
)

# چاپ مجموع RMSLEها (در صورت نیاز به جمع)
print("🔹 RMSLE (sum over families):", rmsle_per_family.sum())

# نمایش RMSLE هر family جداگانه
rmsle_per_family


In [ ]:
y_target_stacked

In [ ]:
# Start and end date for test data
stest = '2017-08-16'
etest = '2017-08-31'

X_test = dp.out_of_sample(steps=16)
X_test['NewYear'] = (X_test.index.dayofyear == 1)

sales_pred = pd.DataFrame(model.predict(X_test), index = X_test.index,columns = y.columns)  
sales_pred = sales_pred.stack(['store_nbr', 'family'])

In [ ]:
# Submit results to competition
df_sub = pd.read_csv(path + 'sample_submission.csv', index_col='id')
df_sub.sales = sales_pred.values
df_sub.to_csv('submission.csv', index=True)